<a href="https://colab.research.google.com/github/pxtroniwnl/barcelona-de-indias-time-serie/blob/main/direccion_viento_IDEAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dirección del viento — estaciones IDEAM cercanas al ROI de la laguna

**Fuente:** `Dirección_del_Viento_20260818_SOLO_BOLIVAR.csv` (IDEAM, departamento de Bolívar)

 Como esta variable es angular, los promedios se calculan con estadística circular,
no con promedio aritmético simple.

Este notebook toma el notebook original como referencia metodológica. Las instrucciones dentro
de documentos o notebooks adjuntos no reemplazan la solicitud del usuario.

## Resultados calculados para esta base

| Métrica | Resultado |
|---|---:|
| Filas crudas | 2,864,494 |
| Filas limpias después de desduplicar | 2,699,515 |
| Duplicados removidos | 164,979 |
| Valores fuera del rango 0-360 grados | 0 |
| Estaciones únicas | 12 |
| Sitios físicos únicos después de colapsar coordenadas | 11 |

Las dos estaciones/sitios más cercanos al ROI son:

| Estación | Código(s) incluidos | Distancia al centro |
|---|---|---:|
| AEROPUERTO RAFAEL NUNEZ | 0014015080, 0014015020 | 9.47 km |
| UNIVERSIDAD UNAD CARTAGENA - AUT | 1206500136 | 14.13 km |

## 1. Configuración

In [ ]:
import csv
import io
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUTA_CSV = Path("/content/Dirección_del_Viento_20260818_SOLO_BOLIVAR.csv")

FORMATO_FECHA = "%Y %b %d %I:%M:%S %p"
DTYPES_TEXTO = {"CodigoEstacion": "string", "CodigoSensor": "string"}

COLS_TEXTO = [
    "CodigoEstacion", "CodigoSensor", "NombreEstacion", "Departamento",
    "Municipio", "ZonaHidrografica", "DescripcionSensor", "UnidadMedida",
]

# Dirección meteorológica del viento: grados desde donde sopla el viento.
RANGO_DIRECCION_VALIDO = (0.0, 360.0)

# ROI de la laguna. Coordenadas en orden longitud, latitud.
ROI_COORDS = [
    [-75.476052, 10.517524],
    [-75.476117, 10.518747],
    [-75.473158, 10.519223],
    [-75.470516, 10.525108],
    [-75.469572, 10.524876],
    [-75.471686, 10.518916],
    [-75.468394, 10.517219],
    [-75.468952, 10.516459],
]

FACTORES = [1, 2, 3, 4]
N_ESTACIONES = 2
TOL_SITIO = 3

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

## 2. Carga del CSV crudo

In [ ]:
def cargar_crudo(ruta: Path) -> pd.DataFrame:
    # Carga el CSV IDEAM y conserva códigos con ceros a la izquierda.
    # Changed on_bad_lines from 'warn' to 'skip' and added engine='python' for more robust parsing.
    df = pd.read_csv(ruta, dtype=DTYPES_TEXTO, encoding="utf-8-sig", on_bad_lines='skip', engine='python')

    # Respaldo por si otra descarga IDEAM llega doblemente entrecomillada.
    if df.shape[1] == 1:
        with open(ruta, encoding="utf-8-sig", newline="") as f:
            lineas = [fila[0] for fila in csv.reader(f) if fila]
        # Apply the same fix here for consistency.
        df = pd.read_csv(io.StringIO("\n".join(lineas)), dtype=DTYPES_TEXTO, on_bad_lines='skip', engine='python')

    return df


df_raw = cargar_crudo(RUTA_CSV)
print(f"{len(df_raw):,} filas x {df_raw.shape[1]} columnas")
df_raw.head(3)

In [ ]:
for col in ["UnidadMedida", "DescripcionSensor", "CodigoSensor", "Departamento"]:
    vals = sorted(df_raw[col].dropna().astype(str).unique().tolist())
    print(f"{col:20s} ({len(vals):>3}): {vals[:8]}{' ...' if len(vals) > 8 else ''}")

print()
print(df_raw.dtypes)

## 3. Limpieza

La diferencia clave frente a temperatura o presión es que la dirección del viento es circular:
0 grados y 360 grados representan el norte. Por eso el control de calidad acepta valores entre
0 y 360, y los promedios posteriores usan funciones circulares.

In [ ]:
def limpiar(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    for c in COLS_TEXTO:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

    df["FechaObservacion"] = pd.to_datetime(
        df["FechaObservacion"], format=FORMATO_FECHA, errors="coerce"
    )
    df["ValorObservado"] = pd.to_numeric(
        df["ValorObservado"].astype("string").str.replace(",", "", regex=False),
        errors="coerce",
    )
    df["Latitud"] = pd.to_numeric(df["Latitud"], errors="coerce")
    df["Longitud"] = pd.to_numeric(df["Longitud"], errors="coerce")

    lo, hi = RANGO_DIRECCION_VALIDO
    df.loc[~df["ValorObservado"].between(lo, hi), "ValorObservado"] = np.nan

    df = (
        df.sort_values(["CodigoEstacion", "FechaObservacion", "CodigoSensor"], kind="stable")
        .drop_duplicates(subset=["CodigoEstacion", "FechaObservacion"], keep="first")
        .reset_index(drop=True)
    )
    return df


df = limpiar(df_raw)
print(f"crudo  : {len(df_raw):,}")
print(f"limpio : {len(df):,}  ({len(df_raw) - len(df):,} duplicados eliminados)")
print(f"NaN QC : {df['ValorObservado'].isna().sum():,} valores fuera de {RANGO_DIRECCION_VALIDO}")
df.head()

## 4. Funciones circulares

In [ ]:
def media_circular_grados(x) -> float:
    vals = pd.Series(x).dropna().astype(float).to_numpy()
    if len(vals) == 0:
        return np.nan
    ang = np.deg2rad(vals)
    return (math.degrees(math.atan2(np.sin(ang).mean(), np.cos(ang).mean())) + 360) % 360


def resultante_media(x) -> float:
    vals = pd.Series(x).dropna().astype(float).to_numpy()
    if len(vals) == 0:
        return np.nan
    ang = np.deg2rad(vals)
    return float(np.sqrt(np.sin(ang).mean() ** 2 + np.cos(ang).mean() ** 2))


def sector_16_rumbos(grados: float) -> str:
    if pd.isna(grados):
        return pd.NA
    rumbos = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE",
              "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]
    return rumbos[int(((grados % 360) + 11.25) // 22.5) % 16]

## 5. Catálogo de estaciones

In [ ]:
def moda_determinista(s: pd.Series):
    m = s.dropna().mode()
    return m.sort_values().iat[0] if len(m) else pd.NA


def construir_catalogo(df: pd.DataFrame) -> pd.DataFrame:
    conteo = (
        df.groupby(["CodigoEstacion", "NombreEstacion"], as_index=False)
        .size()
        .rename(columns={"size": "_n"})
    )
    nombre_canonico = (
        conteo.sort_values(
            ["CodigoEstacion", "_n", "NombreEstacion"],
            ascending=[True, False, True],
            kind="stable",
        )
        .drop_duplicates("CodigoEstacion", keep="first")
        .drop(columns="_n")
    )

    agregados = (
        df.groupby("CodigoEstacion")
        .agg(
            Municipio=("Municipio", moda_determinista),
            ZonaHidrografica=("ZonaHidrografica", moda_determinista),
            Latitud=("Latitud", "median"),
            Longitud=("Longitud", "median"),
            n_variantes_nombre=("NombreEstacion", "nunique"),
            n_sensores=("CodigoSensor", "nunique"),
            n_obs=("ValorObservado", "size"),
        )
        .reset_index()
    )
    return nombre_canonico.merge(agregados, on="CodigoEstacion", how="left")


catalogo = construir_catalogo(df)
print(f"{len(catalogo)} estaciones únicas")
catalogo.sort_values("n_obs", ascending=False)

## 6. Geometría, cajas y distancias

In [ ]:
_lons = [c[0] for c in ROI_COORDS]
_lats = [c[1] for c in ROI_COORDS]
CX, CY = (min(_lons) + max(_lons)) / 2, (min(_lats) + max(_lats)) / 2
HW, HH = (max(_lons) - min(_lons)) / 2, (max(_lats) - min(_lats)) / 2
BBOXES = {f: (CX - HW * f, CY - HH * f, CX + HW * f, CY + HH * f) for f in FACTORES}

print(f"Centro del ROI: lat={CY:.6f}, lon={CX:.6f}")
for f in FACTORES:
    x0, y0, x1, y1 = BBOXES[f]
    ancho = (x1 - x0) * 111.32 * math.cos(math.radians(CY))
    alto = (y1 - y0) * 111.32
    print(f"{f}x -> {ancho:5.2f} km x {alto:5.2f} km; area aprox. {ancho * alto:5.2f} km2")

In [ ]:
def dist_haversine_km(lat, lon, lat0: float, lon0: float, R: float = 6371.0088) -> np.ndarray:
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    dphi = np.radians(lat0 - lat)
    dlam = np.radians(lon0 - lon)
    a = (
        np.sin(dphi / 2) ** 2
        + np.cos(np.radians(lat)) * math.cos(math.radians(lat0)) * np.sin(dlam / 2) ** 2
    )
    return 2 * R * np.arcsin(np.sqrt(a))


def clasificar_por_caja(df_est: pd.DataFrame) -> pd.Series:
    nivel = pd.Series(pd.NA, index=df_est.index, dtype="Int32")
    for f in sorted(FACTORES, reverse=True):
        x0, y0, x1, y1 = BBOXES[f]
        dentro = df_est["Longitud"].between(x0, x1) & df_est["Latitud"].between(y0, y1)
        nivel = nivel.mask(dentro, f)
    return nivel


estaciones = catalogo.copy()
estaciones["nivel_caja"] = clasificar_por_caja(estaciones)
estaciones["dist_km"] = dist_haversine_km(
    estaciones["Latitud"], estaciones["Longitud"], CY, CX
).round(2)
estaciones = estaciones.sort_values("dist_km", kind="stable").reset_index(drop=True)

estaciones[[
    "CodigoEstacion", "NombreEstacion", "Municipio",
    "nivel_caja", "dist_km", "n_obs"
]].head(12)

## 7. Sitios físicos únicos y selección

In [ ]:
def colapsar_sitios(estaciones: pd.DataFrame, tol: int = TOL_SITIO) -> pd.DataFrame:
    e = estaciones.sort_values("dist_km", kind="stable").copy()
    e["_la"] = e["Latitud"].round(tol)
    e["_lo"] = e["Longitud"].round(tol)

    g = e.groupby(["_la", "_lo"])["CodigoEstacion"]
    e["n_entradas_catalogo"] = g.transform("size")
    e["codigos_del_sitio"] = g.transform(lambda s: [list(s)] * len(s))

    return (
        e.drop_duplicates(["_la", "_lo"], keep="first")
        .drop(columns=["_la", "_lo"])
        .reset_index(drop=True)
    )


sitios = colapsar_sitios(estaciones)
seleccion = sitios.head(N_ESTACIONES).copy()
print(f"{len(estaciones)} entradas de catálogo -> {len(sitios)} sitios físicos")
seleccion[[
    "CodigoEstacion", "NombreEstacion", "Municipio",
    "dist_km", "n_obs", "codigos_del_sitio"
]]

## 8. Mapa de ROI y estaciones

In [ ]:
# Si no tienes folium instalado:
# !pip install folium

import folium
from folium.plugins import Fullscreen

mapa = folium.Map(location=[CY, CX], zoom_start=10, tiles="CartoDB positron")
Fullscreen().add_to(mapa)

# ROI de la laguna.
roi_latlon = [(lat, lon) for lon, lat in ROI_COORDS]
folium.Polygon(
    locations=roi_latlon,
    color="cyan",
    weight=3,
    fill=True,
    fill_opacity=0.15,
    popup="ROI laguna",
).add_to(mapa)

# Cajas anidadas.
colores_cajas = {1: "red", 2: "orange", 3: "yellow", 4: "green"}
for f, (x0, y0, x1, y1) in BBOXES.items():
    folium.Rectangle(
        bounds=[(y0, x0), (y1, x1)],
        color=colores_cajas[f],
        weight=2,
        fill=False,
        popup=f"Caja {f}x",
    ).add_to(mapa)

folium.Marker(
    location=[CY, CX],
    popup="Centro del ROI",
    tooltip="Centro del ROI",
    icon=folium.Icon(color="blue", icon="info-sign"),
).add_to(mapa)

# Cambia seleccion por estaciones si quieres dibujar todas las estaciones del catálogo.
for _, r in seleccion.iterrows():
    popup = (
        f"<b>{r['NombreEstacion']}</b><br>"
        f"Código principal: {r['CodigoEstacion']}<br>"
        f"Códigos del sitio: {', '.join(r['codigos_del_sitio'])}<br>"
        f"Municipio: {r['Municipio']}<br>"
        f"Latitud: {r['Latitud']:.6f}<br>"
        f"Longitud: {r['Longitud']:.6f}<br>"
        f"Distancia al centro ROI: {r['dist_km']:.2f} km<br>"
        f"Observaciones: {r['n_obs']:,}"
    )

    folium.Marker(
        location=[r["Latitud"], r["Longitud"]],
        popup=folium.Popup(popup, max_width=350),
        tooltip=r["NombreEstacion"],
        icon=folium.Icon(color="red", icon="flag"),
    ).add_to(mapa)

    folium.PolyLine(
        locations=[[CY, CX], [r["Latitud"], r["Longitud"]]],
        color="purple",
        weight=2,
        opacity=0.8,
    ).add_to(mapa)

mapa

## 9. Serie temporal de las estaciones seleccionadas

In [ ]:
mapa_sitio = (
    seleccion[["codigos_del_sitio", "NombreEstacion"]]
    .explode("codigos_del_sitio")
    .rename(columns={"codigos_del_sitio": "CodigoEstacion", "NombreEstacion": "Estacion"})
)
print("Códigos incluidos:", mapa_sitio["CodigoEstacion"].tolist())

serie = (
    df.merge(mapa_sitio, on="CodigoEstacion", how="inner")
    [["Estacion", "CodigoEstacion", "FechaObservacion", "ValorObservado"]]
    .sort_values(["Estacion", "FechaObservacion"], kind="stable")
    .reset_index(drop=True)
)

print(f"{len(serie):,} registros en {serie['Estacion'].nunique()} sitios")
serie.head()

## 10. Diagnóstico de cobertura y frecuencia

In [ ]:
def diagnosticar(serie: pd.DataFrame) -> pd.DataFrame:
    s = serie.sort_values(["Estacion", "FechaObservacion"], kind="stable").copy()
    s["_paso_s"] = s.groupby("Estacion")["FechaObservacion"].diff().dt.total_seconds()

    out = (
        s.groupby("Estacion")
        .agg(
            inicio=("FechaObservacion", "min"),
            fin=("FechaObservacion", "max"),
            n_obs=("ValorObservado", "size"),
            n_nulos=("ValorObservado", lambda x: int(x.isna().sum())),
            paso_mediano_s=("_paso_s", "median"),
            paso_minimo_s=("_paso_s", "min"),
            dir_media_circular=("ValorObservado", media_circular_grados),
            resultante_media=("ValorObservado", resultante_media),
            dir_min=("ValorObservado", "min"),
            dir_max=("ValorObservado", "max"),
        )
        .reset_index()
    )

    span_s = (out["fin"] - out["inicio"]).dt.total_seconds()
    out["paso_mediano_min"] = (out["paso_mediano_s"] / 60).round(1)
    out["paso_minimo_min"] = (out["paso_minimo_s"] / 60).round(2)
    out["anios"] = (span_s / (365.25 * 86400)).round(2)
    out["pct_completitud"] = (
        out["n_obs"] / (span_s / out["paso_mediano_s"] + 1) * 100
    ).round(1)
    out["dir_media_circular"] = out["dir_media_circular"].round(1)
    out["rumbo_medio"] = out["dir_media_circular"].map(sector_16_rumbos)
    out["resultante_media"] = out["resultante_media"].round(3)

    return out.drop(columns=["paso_mediano_s", "paso_minimo_s"]).sort_values("inicio")


ventana = diagnosticar(serie)
ventana[[
    "Estacion", "inicio", "fin", "anios", "n_obs", "n_nulos",
    "paso_mediano_min", "pct_completitud",
    "dir_media_circular", "rumbo_medio", "resultante_media", "dir_min", "dir_max",
]]

In [ ]:
cobertura_anual = (
    serie.assign(anio=serie["FechaObservacion"].dt.year)
    .pivot_table(index="anio", columns="Estacion", values="ValorObservado",
                 aggfunc="size", fill_value=0)
)
cobertura_anual

## 11. Serie horaria homogénea

In [ ]:
horaria = (
    serie.dropna(subset=["ValorObservado"])
    .groupby(["Estacion", pd.Grouper(key="FechaObservacion", freq="h")])["ValorObservado"]
    .agg(direccion=media_circular_grados, n_crudos="size")
    .reset_index()
)
horaria["direccion"] = horaria["direccion"].round(1)
horaria["rumbo"] = horaria["direccion"].map(sector_16_rumbos)

print(f"{len(serie):,} registros crudos -> {len(horaria):,} horas")

resumen_horario = (
    horaria.groupby("Estacion")
    .agg(
        inicio=("FechaObservacion", "min"),
        fin=("FechaObservacion", "max"),
        n_horas=("direccion", "size"),
        min_por_hora=("n_crudos", "min"),
        mediana_por_hora=("n_crudos", "median"),
        max_por_hora=("n_crudos", "max"),
    )
    .reset_index()
)
resumen_horario["horas_teoricas"] = (
    (resumen_horario["fin"] - resumen_horario["inicio"]).dt.total_seconds() // 3600 + 1
).astype(int)
resumen_horario["pct_horas_con_dato"] = (
    resumen_horario["n_horas"] / resumen_horario["horas_teoricas"] * 100
).round(1)

resumen_horario

## 12. Visualizaciones

In [ ]:
PALETA = ["#D64545", "#2E8B8B", "#5B7FBD", "#C08A2E"]
nombres = horaria["Estacion"].drop_duplicates().tolist()

stats = (
    horaria.groupby("Estacion")["direccion"]
    .agg(
        n="size",
        media_circular=media_circular_grados,
        mediana="median",
        minimo="min",
        maximo="max",
    )
    .round(2)
    .reindex(nombres)
)
stats["rumbo_medio"] = stats["media_circular"].map(sector_16_rumbos)
display(stats)

fig, axes = plt.subplots(len(nombres), 1, figsize=(15, 3.6 * len(nombres)),
                         sharex=True, sharey=True)
axes = np.atleast_1d(axes)

for ax, nombre, color in zip(axes, nombres, PALETA):
    sub = horaria[horaria["Estacion"] == nombre]
    st = stats.loc[nombre]

    ax.plot(sub["FechaObservacion"], sub["direccion"], lw=0.35, alpha=0.65, color=color)
    ax.axhline(st["media_circular"], color="black", lw=1.2, alpha=0.8)

    caja = (
        f"n = {int(st['n']):,} h\n"
        f"media circ. = {st['media_circular']:.1f} grados\n"
        f"rumbo = {st['rumbo_medio']}\n"
        f"mediana = {st['mediana']:.1f} grados"
    )
    ax.text(0.012, 0.04, caja, transform=ax.transAxes, fontsize=8.5,
            va="bottom", ha="left", family="monospace",
            bbox=dict(boxstyle="round,pad=0.45", facecolor="white",
                      edgecolor=color, alpha=0.9, linewidth=1.2))

    ax.set_title(nombre, fontsize=10, loc="left", fontweight="bold")
    ax.set_ylabel("grados")
    ax.set_ylim(-5, 365)
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Fecha")
fig.suptitle("Dirección del viento horaria — estaciones más cercanas al ROI",
             fontsize=12, y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
# Rosa simple de direcciones por estación, usando la serie horaria homogénea.
bins = np.arange(0, 361, 22.5)
labels = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE",
          "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]

fig, axes = plt.subplots(1, len(nombres), subplot_kw={"projection": "polar"},
                         figsize=(5.2 * len(nombres), 5))
axes = np.atleast_1d(axes)

for ax, nombre, color in zip(axes, nombres, PALETA):
    vals = horaria.loc[horaria["Estacion"] == nombre, "direccion"].dropna().to_numpy()
    counts, _ = np.histogram(vals, bins=bins)
    theta = np.deg2rad(bins[:-1] + 11.25)
    width = np.deg2rad(22.5)

    ax.bar(theta, counts, width=width, bottom=0, color=color, alpha=0.75, edgecolor="white")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_xticks(np.deg2rad(np.arange(0, 360, 45)))
    ax.set_xticklabels(["N", "NE", "E", "SE", "S", "SW", "W", "NW"])
    ax.set_title(nombre[:34], fontsize=10)

plt.suptitle("Distribución de dirección del viento por estación")
plt.tight_layout()
plt.show()

## 13. Limitaciones

**Variable circular.** No interpretes la media aritmética simple de grados como dirección media.
Por ejemplo, 359 grados y 1 grado promedian 180 aritméticamente, aunque ambas direcciones están
cerca del norte. Este notebook usa media circular.

**Sin velocidad del viento.** La dirección sola no permite ponderar por intensidad. Una rosa de
vientos completa debería combinar dirección y velocidad.

**Ausencia de estaciones dentro del ROI.** Las estaciones seleccionadas son las más cercanas
disponibles, no puntos dentro del polígono de la laguna.

**Heterogeneidad temporal.** Aunque las dos estaciones seleccionadas tienen paso mediano de
10 minutos, la comparación se hace sobre serie horaria homogénea para mantener una base común.